In [ ]:
# OBSERVAÇÕES : 
# VERSÃO DO PYTHON: 3.14.3
# Bibliotecas: Listadas A baixo voce precissa fazer o PIP install delas para que suas funções funcione
# Repositorio do Projeto : https://github.com/Ryanlssv/coding_test
import pandas as pd
import matplotlib.pyplot as plt
import math
import requests
import numpy as np
import statsmodels.api as sm
# Biblioteca tabulate para visualizar os dados como uma planilha real
from tabulate import tabulate

# Questão 1 – Tratamento de dados.
print("\n-- Questão 1 – Tratamento de dados")
#A) Leia os arquivos XLSX em anexo. Construa um dataframe, usando pandas, que contenha todos os ativos da base de 
# classes locais (dados_locais.xlsx) e os ativos da classe offshore (dados_offshore.xlsx). O merge precisa ser pela Data. 

print("\n-- A) Leia os arquivos XLSX em anexo. Construa um dataframe, usando pandas, que contenha todos os ativos da base de classes locais (dados_locais.xlsx) e os ativos da classe offshore (dados_offshore.xlsx). O merge precisa ser pela Data. --\n")

# Variavel para carregar ambos os arquivos (locais e offshore).
dados1 = pd.read_excel('retornos_dados_locais.xlsx',skiprows=1)
dados2 = pd.read_excel('retornos_dados_offshore.xlsx',skiprows=1)

# Ajustando nomes e tipos
dados1.rename(columns={dados1.columns[0]: 'Data'}, inplace=True)
dados2.rename(columns={dados2.columns[0]: 'Data'}, inplace=True)

# Variavel para Armazenar o merge das tabeleas
dados_final = pd.merge(dados1, dados2, on='Data', how='outer')

#Converter a coluna Data para o formato datetime real
dados_final['Data'] = pd.to_datetime(dados_final['Data'], errors='coerce')

#Converter todas as outras colunas para numérico (float)
colunas_ativos = dados_final.columns.drop('Data')
dados_final[colunas_ativos] = dados_final[colunas_ativos].apply(pd.to_numeric, errors='coerce')

print(tabulate(dados_final.head(), headers='keys', tablefmt='psql'))


-- Questão 1 – Tratamento de dados

-- A) Leia os arquivos XLSX em anexo. Construa um dataframe, usando pandas, que contenha todos os ativos da base de classes locais (dados_locais.xlsx) e os ativos da classe offshore (dados_offshore.xlsx). O merge precisa ser pela Data. --

+----+---------------------+-------------+-------------+-------------+-------------------+-------------+--------------+--------------+-------------+--------------+-------------+-------------+
|    | Data                |         CDI |       Dólar |    Ibovespa |   IDkA Pré 3 Anos |        IFIX |         IHFA |        IMA-B |     IMA-B 5 |     IMA-B 5+ |   RV Global |   RF Global |
|----+---------------------+-------------+-------------+-------------+-------------------+-------------+--------------+--------------+-------------+--------------+-------------+-------------|
|  0 | 2015-01-02 00:00:00 | 0.000434547 |  0.0138167  | -0.0298958  |       0.00117785  |  0.00733138 |  0.00216921  |  0.000883892 | 0.000800841 

In [39]:
#B) Dado os pesos que temos na carteira recomendada MODERADA* na nossa carta mensal,
#crie 3 portfólios que diferementre si pelo ativo que representa a parcela de inflação**. Ou seja,


print("\n-- B) Dado os pesos que temos na carteira recomendada MODERADA* na nossa carta mensal --\n")
# Guandando as taxas nas respectivas variaveis
peso_RFI = 10 / 100
peso_ALTL = 2 / 100
peso_PosFixa = 31.5 / 100
peso_Inflacao = 22.5 / 100
peso_RetAb = 16.5 / 100
peso_RvLocal = 5 / 100
peso_RVglob = 7.5 / 100
peso_Flls = 5 / 100

# Criando variavel de ativos presentes nos 3 portifolios 
fixos = ( 
  (peso_RFI * dados_final['RF Global']) 
+ (peso_ALTL * dados_final['Dólar']) 
+ (peso_PosFixa * dados_final['CDI']) 
+ (peso_RetAb * dados_final['IHFA']) 
+ (peso_RvLocal * dados_final['Ibovespa']) 
+ (peso_RVglob * dados_final['RV Global'])
+ (peso_Flls * dados_final['IFIX'])
)
# Criando variaveis para o IMA-B nos 3 portifolios 
dados_final['Portifolio_1'] = (fixos + peso_Inflacao * dados_final['IMA-B'])
dados_final['Portifolio_2'] = (fixos + peso_Inflacao * dados_final['IMA-B 5'])
dados_final['Portifolio_3'] = (fixos + peso_Inflacao * dados_final['IMA-B 5+'])


# Cálculo do retorno acumulado total
acumulado = (1 + dados_final[['Portifolio_1', 'Portifolio_2', 'Portifolio_3']]).prod() - 1

# Criando uma lista para o tabulate
resumo = [
    ["Portfólio 1 (IMA-B)", acumulado['Portifolio_1']],
    ["Portfólio 2 (IMA-B 5)", acumulado['Portifolio_2']],
    ["Portfólio 3 (IMA-B 5+)", acumulado['Portifolio_3']]
]

print(tabulate(resumo, headers=["Estratégia", "Retorno Total"], tablefmt='psql', stralign="left", numalign="right", floatfmt=".2%"))



-- B) Dado os pesos que temos na carteira recomendada MODERADA* na nossa carta mensal --

+------------------------+-----------------+
| Estratégia             |   Retorno Total |
|------------------------+-----------------|
| Portfólio 1 (IMA-B)    |         229.43% |
| Portfólio 2 (IMA-B 5)  |         228.80% |
| Portfólio 3 (IMA-B 5+) |         229.44% |
+------------------------+-----------------+


In [3]:
#C)
print("-- C) --")
#a) Retorno total e retorno total anualizado de cada portfólio

print("-- a) Retorno total e retorno total anualizado de cada portfólio --\n")

# Tempo em anos
cagr = (dados_final['Data'].max() - dados_final['Data'].min()).days / 365

RT_Atualizado = (1 + acumulado) ** (1 / cagr) - 1

resumo = [
    ['Portifolio 1', acumulado['Portifolio_1'],RT_Atualizado['Portifolio_1']],
    ['Portifolio 2', acumulado['Portifolio_2'],RT_Atualizado['Portifolio_2']],
    ['Portifolio 3', acumulado['Portifolio_3'],RT_Atualizado['Portifolio_3']]
]

print(tabulate(resumo,headers=["Portfólio", "Retorno Total", "Retorno Anualizado"],floatfmt=".2%"))


#b) Volatilidade total e volatilidade total anualizada de cada portfólio
print("\n-- b) Volatilidade total e volatilidade total anualizada de cada portfólio --\n")

retornos = dados_final[['Portifolio_1', 'Portifolio_2', 'Portifolio_3']]
#Usando o comando std() do pandas para calcular o desvio padão
Volatividade = retornos.std() 

#Montando a saida com tabulate
resumo = [
    ['Portifolio 1',Volatividade['Portifolio_1']],
    ['Portifolio 2',Volatividade['Portifolio_2']],
    ['Portifolio 3',Volatividade['Portifolio_3']]
]
print(tabulate(resumo,headers=["Portifolio","Volatilidade Total"],tablefmt='psql', stralign="left", numalign="right" ,floatfmt=".2%"))

#Calculando a Volativiade Anualizada
# volativiade_Anualizada = Volatividade * (raiz quadrada) de 252
dias_Util = math.sqrt(252)
Vt_Anualizada = Volatividade * dias_Util

#Montando a saida com tabulate
resumo2 = [
    ['Portifolio 1',Vt_Anualizada['Portifolio_1']],
    ['Portifolio 2',Vt_Anualizada['Portifolio_2']],
    ['Portifolio 3',Vt_Anualizada['Portifolio_3']]
]

print(tabulate(resumo2,headers=["Portifolio","Volatilidade Anualizada"],tablefmt='psql', stralign="left", numalign="right" ,floatfmt=".2%"))


#C) Fazer o Calculo do Sharpe= ​Rp​−Rf​​ / op

print("\n-- c) Fazer o Calculo do Sharpe= ​Rp​−Rf​​ / op --\n")
# Calcular retorno acumulado do CDI e transformar em anualizado
rf_acumulado = (1 + dados_final['CDI']).prod() - 1
anos = (dados_final['Data'].max() - dados_final['Data'].min()).days / 365
rf_anual = (1 + rf_acumulado) ** (1 / anos) - 1


rp = cagr / 100
op= Vt_Anualizada
Sharpe = (rp - rf_anual)/op

resumo3 = [
    ['Portifolio 1',Sharpe['Portifolio_1']],
    ['Portifolio 2',Sharpe['Portifolio_2']],
    ['Portifolio 3',Sharpe['Portifolio_3']]
]

print(tabulate(resumo3,headers=["Portifolio","Sharpe"],tablefmt='psql', stralign="left", numalign="right"))




-- C) --
-- a) Retorno total e retorno total anualizado de cada portfólio --

Portfólio       Retorno Total    Retorno Anualizado
------------  ---------------  --------------------
Portifolio 1          229.43%                11.44%
Portifolio 2          228.80%                11.42%
Portifolio 3          229.44%                11.44%

-- b) Volatilidade total e volatilidade total anualizada de cada portfólio --

+--------------+----------------------+
| Portifolio   |   Volatilidade Total |
|--------------+----------------------|
| Portifolio 1 |                0.22% |
| Portifolio 2 |                0.20% |
| Portifolio 3 |                0.25% |
+--------------+----------------------+
+--------------+---------------------------+
| Portifolio   |   Volatilidade Anualizada |
|--------------+---------------------------|
| Portifolio 1 |                     3.54% |
| Portifolio 2 |                     3.12% |
| Portifolio 3 |                     4.02% |
+--------------+----------------

In [4]:
print("\n-- Questão 2 – Com os mesmos portfólios da questão anterior, faça o que se pede. --\n")
print("\n--A) Construa um módulo em Python com as seguintes funções: --\n")
#a
print("\n-- a) Semidesvio dos portfólios --\n")
def semidesvio (retornos):
    negativos = retornos[retornos < 0]
    return negativos.std()
sd = semidesvio(retornos)

resumo = [
    ["Portfólio 1", sd['Portifolio_1']],
    ["Portfólio 2", sd['Portifolio_2']],
    ["Portfólio 3", sd['Portifolio_3']]
]

print(tabulate(resumo, headers=["Portfólio", "Semidesvio"], floatfmt=".2%"))

#b
print("\n-- b) Máximo Drawdown dos portfólios --\n")

retornos = dados_final[['Portifolio_1', 'Portifolio_2', 'Portifolio_3']]
def max_drawdown(retornos):
    acumulado = (1 + retornos).cumprod()
    pico = acumulado.cummax()
    drawdown = (acumulado / pico) - 1
    return drawdown.min()

mdd = max_drawdown(retornos)

resumo = [
    ["Portfólio 1", mdd['Portifolio_1']],
    ["Portfólio 2", mdd['Portifolio_2']],
    ["Portfólio 3", mdd['Portifolio_3']]
]

print(tabulate(resumo, headers=["Portfólio", "Max Drawdown"], floatfmt=".2%"))

#c
print("\n-- c) VaR 95 Paramétrico dos portfólios --\n")
def var_parametrico(retornos, nivel=0.95):
    z = -1.65
    media = retornos.mean()
    vol = retornos.std()
    return media + z * vol
var_p = var_parametrico(retornos)

resumo = [
    ["Portfólio 1", var_p['Portifolio_1']],
    ["Portfólio 2", var_p['Portifolio_2']],
    ["Portfólio 3", var_p['Portifolio_3']]
]

print(tabulate(resumo, headers=["Portfólio", "VaR Paramétrico"], floatfmt=".2%"))

#d
print("\n-- d) VaR 95 Histórico dos portfólios --\n")
def var_historico(retornos):
    return retornos.quantile(0.05)

var_h = var_historico(retornos)

resumo = [
    ["Portfólio 1", var_h['Portifolio_1']],
    ["Portfólio 2", var_h['Portifolio_2']],
    ["Portfólio 3", var_h['Portifolio_3']]
]

print(tabulate(resumo, headers=["Portfólio", "VaR Histórico"], floatfmt=".2%"))

#e
print("\n-- e) Expected Shortfall (ES 95) dos portfólios --\n")
def expected_shortfall(retornos, nivel=0.95):
    var = retornos.quantile(1 - nivel)
    es = {}

    for col in retornos.columns:
        piores = retornos[col][retornos[col] <= var[col]]
        es[col] = piores.mean()

    return pd.Series(es)

es = expected_shortfall(retornos)

resumo = [
    ["Portfólio 1", es['Portifolio_1']],
    ["Portfólio 2", es['Portifolio_2']],
    ["Portfólio 3", es['Portifolio_3']]
]

print(tabulate(resumo, headers=["Portfólio", "ES 95%"], floatfmt=".2%"))







-- Questão 2 – Com os mesmos portfólios da questão anterior, faça o que se pede. --


--A) Construa um módulo em Python com as seguintes funções: --


-- a) Semidesvio dos portfólios --

Portfólio      Semidesvio
-----------  ------------
Portfólio 1         0.20%
Portfólio 2         0.17%
Portfólio 3         0.23%

-- b) Máximo Drawdown dos portfólios --

Portfólio      Max Drawdown
-----------  --------------
Portfólio 1         -10.41%
Portfólio 2          -8.86%
Portfólio 3         -11.75%

-- c) VaR 95 Paramétrico dos portfólios --

Portfólio      VaR Paramétrico
-----------  -----------------
Portfólio 1             -0.32%
Portfólio 2             -0.28%
Portfólio 3             -0.37%

-- d) VaR 95 Histórico dos portfólios --

Portfólio      VaR Histórico
-----------  ---------------
Portfólio 1           -0.24%
Portfólio 2           -0.23%
Portfólio 3           -0.28%

-- e) Expected Shortfall (ES 95) dos portfólios --

Portfólio      ES 95%
-----------  --------
Portfólio 1    

In [ ]:
print("\n-- b) Escreva em suas palavras quais seriam as vantagens e desvantagens de se usar o VaR Paramétrico ou o histórico e em quais cenários/contexto seria vantajoso escolher cada uma. --\n")

print("""
R:  O VaR Paramétrico consegue interpretar retornos de ativos seguindo uma distribuição de probabilidade conhecida, com isso possui uma agilidade computacional
eficiente e rápida. Sua simplicidade e uma vantagem de uso pois precisa de dois parâmetros principais para o cálculo.

\n""")

print("\n-- c) Explique em suas palavras o que é o semidesvio e qual a vantagem de usar ele em análises de risco. --\n")

print("""
R: O semidesvio é uma forma de calcular a volatilidade de um ativo no mercado financeiro, focando apenas nos retornos que ficam abaixo da média ou abaixo de
um retorno alvo específico. As vantagens de usar o semidesvio são: foco real no risco de perda, o semidesvio é considerado uma métrica superior ao desvio padrão
para investidores avessos ao risco, pois ele ignora a "volatilidade positiva".
\n""")


print("\n-- d) Explique em suas palavras o que seria o Expected Shortfall e sua vantagem de usar essa métrica em análise de risco. --\n")

print("""
R: O Expected Shortfall é uma medida de risco que mostra a média das piores perdas de um investimento.
Ou seja, ele indica quanto você pode perder em situações muito ruins.A vantagem é que ele é mais completo que o VaR, 
porque não mostra só o limite da perda, mas também o tamanho das perdas mais graves.
\n""")


print("\n-- e) Explique em suas palavras a diferença, na linguagem Python, entre um módulo, uma biblioteca e um pacote. --\n")

print("""
R: O Expected Shortfall é uma medida de risco que mostra a média das piores perdas de um investimento. Sua vantagem é que ele é mais completo que o VaR, 
pois mostra não só o limite da perda, mas também o tamanho das perdas mais graves. Em Python, um módulo é um arquivo com código, uma biblioteca é um conjunto 
de módulos, e um pacote é uma forma de organizar esses módulos em pastas.
""")



In [5]:
print("\n-- Questão 3 – Coletar dados públicos via AP --\n")
print("\n-- a) O Banco Central possui um portal de dados abertos em que disponibiliza a séria histórica do CDI, da SELIC e outros dadosvia API. Usando a biblioteca pandas e a função read_json, colete a série histórica do CDI com início em 01/01/2010. --\n")

# criando Função para implementar dados da API Refente a Taxa de juros diária (CDI / taxa interbancária diária)
# A Api do banco central somente pega dados de até 10 anos do periodo selecionado nao ultrapasando este valor,
# então devemos quebrar em 2 partes para pegados todos os dados ate a hoje
def pegar_dados_CDI(inicio,fim):
        url = f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.11/dados?formato=json&dataInicial={inicio}&dataFinal={fim}'
        response = requests.get(url)
        dados = response.json()
        return pd.DataFrame(dados)
# Periodo de busca dos dados 
df1 = pegar_dados_CDI('01/01/2010', '31/12/2019')
df2 = pegar_dados_CDI('01/01/2020', '31/12/2025')

df = pd.concat([df1,df2])

df['data'] = pd.to_datetime(df['data'], dayfirst=True)
df['valor'] = pd.to_numeric(df['valor'])

print("\n-- Taxa de juros diária (CDI / taxa interbancária diária) --\n")
print(tabulate(df.head(),headers=["Data","Valor"],tablefmt='psql',floatfmt=".2%"))
print(tabulate(df.tail(),headers=["Data","Valor"],tablefmt='psql',floatfmt=".2%"))




-- Questão 3 – Coletar dados públicos via AP --


-- a) O Banco Central possui um portal de dados abertos em que disponibiliza a séria histórica do CDI, da SELIC e outros dadosvia API. Usando a biblioteca pandas e a função read_json, colete a série histórica do CDI com início em 01/01/2010. --


-- Taxa de juros diária (CDI / taxa interbancária diária) --

+----+---------------------+---------+
|    | Data                |   Valor |
|----+---------------------+---------|
|  0 | 2010-01-04 00:00:00 |   3.29% |
|  1 | 2010-01-05 00:00:00 |   3.29% |
|  2 | 2010-01-06 00:00:00 |   3.29% |
|  3 | 2010-01-07 00:00:00 |   3.29% |
|  4 | 2010-01-08 00:00:00 |   3.29% |
+----+---------------------+---------+
+------+---------------------+---------+
|      | Data                |   Valor |
|------+---------------------+---------|
| 1502 | 2025-12-24 00:00:00 |   5.51% |
| 1503 | 2025-12-26 00:00:00 |   5.51% |
| 1504 | 2025-12-29 00:00:00 |   5.51% |
| 1505 | 2025-12-30 00:00:00 |   5.51% |
| 

In [ ]:
print("\n-- b) Buscar fundo na base da CVM e criar um grafico --\n")


df = pd.read_csv('inf_diario_fi_202603.csv', sep=';', encoding='latin1')

# buscando o fundo na base de dados
CNPJ_FUNDO = "36.671.748/0001-63" 
df_fundo = df[df['CNPJ_FUNDO_CLASSE'] == CNPJ_FUNDO].copy()

# Converter data
df_fundo['DT_COMPTC'] = pd.to_datetime(df_fundo['DT_COMPTC'])

# Ordenar
df_fundo = df_fundo.sort_values('DT_COMPTC')


df_fundo['RET_FUNDO'] = df_fundo['VL_QUOTA'].pct_change()
df_fundo['RET_ACUM_FUNDO'] = (1 + df_fundo['RET_FUNDO']).cumprod()


# buscando os dados do banco CDI (Banco Central)

data_inicio = df_fundo['DT_COMPTC'].min().strftime('%d/%m/%Y')
data_fim = df_fundo['DT_COMPTC'].max().strftime('%d/%m/%Y')

url_cdi = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados?formato=json&dataInicial={data_inicio}&dataFinal={data_fim}"
cdi = pd.DataFrame(requests.get(url_cdi).json())

cdi['data'] = pd.to_datetime(cdi['data'], dayfirst=True)
cdi['valor'] = cdi['valor'].astype(float) / 100

# Retorno acumulado CDI
cdi['RET_ACUM_CDI'] = (1 + cdi['valor']).cumprod()


df_merge = pd.merge(
    df_fundo[['DT_COMPTC', 'RET_ACUM_FUNDO']],
    cdi[['data', 'RET_ACUM_CDI']],
    left_on='DT_COMPTC',
    right_on='data',
    how='inner'
)


# Calculando o Retorno total
ret_fundo = df_merge['RET_ACUM_FUNDO'].iloc[-1] - 1
ret_cdi = df_merge['RET_ACUM_CDI'].iloc[-1] - 1

print(f"Retorno Fundo: {ret_fundo:.2%}")
print(f"Retorno CDI: {ret_cdi:.2%}")


# usando a biblioteca do pyplot para gerar o grafico com as informações obtidadas

plt.figure(figsize=(10,6))

plt.plot(df_merge['DT_COMPTC'], df_merge['RET_ACUM_FUNDO'], label='Fundo')
plt.plot(df_merge['DT_COMPTC'], df_merge['RET_ACUM_CDI'], label='CDI')

plt.title('Fundo vs CDI')
plt.xlabel('Data')
plt.ylabel('Retorno acumulado')
plt.legend()
plt.grid()
plt.show()

In [ ]:

print("\n-- Questão 4 – Análise de Regressão. --\n")
print("\n-- A)--\n")
df = pd.read_excel('retornos_dados_locais.xlsx')

# Renomear primeira coluna para Data
df.rename(columns={df.columns[0]: 'Data'}, inplace=True)


# Corrigir DATA

df['Data'] = pd.to_datetime(df['Data'], errors='coerce')
df = df.dropna(subset=['Data'])
df = df.sort_values('Data')


# Corrigir colunas numéricas

for col in df.columns[1:]:
    # Converte tudo pra string primeiro
    df[col] = df[col].astype(str)
    
    # Remove pontos de milhar
    df[col] = df[col].str.replace('.', '', regex=False)
    
    # Troca vírgula por ponto
    df[col] = df[col].str.replace(',', '.', regex=False)
    
    # Converte pra número
    df[col] = pd.to_numeric(df[col], errors='coerce')


df = df.dropna()


# Calcular retornos

df_ret = df.copy()

for col in df.columns[1:]:
    df_ret[col] = df[col].pct_change()

# Substitui infinitos por NaN
df_ret = df_ret.replace([np.inf, -np.inf], np.nan)

# Remove qualquer linha com NaN
df_ret = df_ret.dropna()


# Regressão

Y = df_ret[df.columns[1]]  
X = df_ret[df.columns[2:]]

X = sm.add_constant(X)
# Substitui infinitos por NaN
df_ret = df_ret.replace([np.inf, -np.inf], np.nan)

# Remove qualquer linha com NaN
modelo = sm.OLS(Y, X).fit()

print(modelo.summary())

In [ ]:
print("""\n-- B) Há multicolinearidade, evidenciada pelo alto número de condição. Isso indica forte correlação entre variáveis explicativas.
Pode ser tratada removendo variáveis com alto VIF ou utilizando PCA. --\n""")

print("""C) A regressão foi refeita após a remoção de variáveis com alto VIF, reduzindo a multicolinearidade e tornando os coeficientes mais estáveis.""")